## Skieur Mapping Analysis: MP1 vs MP2 (SS pickles)

All four session types from Google Sheet, loaded as spike-sorted pickles:

| Session type | Condition | Description | Used as |
|-------------|-----------|-------------|---------|
| `tracking_only` | Tracking only | Animal controls frequency, no playback | configurable MP1 |
| `playback` | Tracking + Playback | Standard playback sessions | configurable MP1 |
| `mapping_change_only` | Tracking + Playback | Mapping changed sessions | configurable MP2 |

**Google Sheet is the single source of truth** for session type classification.
Each session's feature DataFrame gets `mapping_type`, `session_type`, `condition_label` columns.


In [ ]:
# ============================================================
# Imports + Config
# ============================================================
import os, sys, pickle, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel, ttest_ind, chi2
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

from utils_load_data import *
from utils_trajectories import *

# ---- Paths ----
save_directory = "./"
NAS = r"\\129.199.81.18\\data5\\eTheremin"

# ---- Parameters ----
dt = 0.005
t_pre, t_post = 0.3, 0.3
time = np.arange(-t_pre, t_post + dt, dt)
n_pre = int(t_pre/dt - 1)

# ---- Plotting ----
mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

# ---- Google Sheet (session type authority) ----
SHEET_URL = ("https://docs.google.com/spreadsheets/d/"
             "1sFatSTXO0j3OONKstz7YN-mM04kNMjk_r7zo951yicU/"
             "gviz/tq?tqx=out:csv&sheet=SKIEUR")
df_sheet = pd.read_csv(SHEET_URL)
df_sheet = df_sheet[df_sheet["use"] == "yes"].copy()
df_sheet["session_name"] = df_sheet["session"].str.strip()

print("Session types in Google Sheet (use=yes only):")
for t in sorted(df_sheet["type"].unique()):
    names = df_sheet[df_sheet["type"] == t]["session_name"].tolist()
    print(f"  {t}: {len(names)} sessions  e.g. {names[:3]}")

print("\nImports and config ready.")


In [ ]:
# ============================================================
# Helpers: load SS pickles + label from Google Sheet
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f:
        n_data = pickle.load(f)
    with open(feat_path, "rb") as f:
        f_data = pickle.load(f)
    return n_data, f_data


def label_sessions(f_data_list, mapping_type, session_type_label):
    "Add mapping_type, session_type, condition_label to each session's features."
    for i, fd in enumerate(f_data_list):
        fd['mapping_type'] = mapping_type
        fd['session_type'] = session_type_label
        if 'Condition' in fd.columns:
            fd['condition_label'] = fd['Condition'].map(
                {0.0: 'Tracking', 1.0: 'Playback', 0: 'Tracking', 1: 'Playback'}).fillna('Unknown')
    return f_data_list


def print_classification(f_data_list, label):
    "Print mapping_type x session_type x condition_label summary."
    all_fd = pd.concat([fd[['mapping_type', 'session_type', 'condition_label']]
                         for fd in f_data_list], ignore_index=True)
    print(f"\n  {label} Classification:")
    for (mt, st, cl), count in all_fd.groupby(
        ['mapping_type', 'session_type', 'condition_label']).size().items():
        print(f"    {mt:5s} | {st:20s} | {cl:9s} : {count:>8,} timepoints")


print("Helpers ready.")


---
## Load Data

Pick which two session types to compare as MP1 and MP2.
All four types are available — change `B_MP1_TYPE` / `B_MP2_TYPE` to any pair from the Sheet output above.


In [ ]:
# ============================================================
# Choose MP1 / MP2 session types
# ============================================================
B_MP1_TYPE = 'tracking_only'         # MP1: Tracking only (no PB)
B_MP2_TYPE = 'mapping_change_only'   # MP2: Tracking + Playback

mp1_label = 'MP1 (' + B_MP1_TYPE + ')'
mp2_label = 'MP2 (' + B_MP2_TYPE + ')'

for label, stype in [(mp1_label, B_MP1_TYPE), (mp2_label, B_MP2_TYPE)]:
    n = len(df_sheet[df_sheet['type'] == stype])
    print(f"  {label}: {stype} -> {n} sessions in Google Sheet")


In [ ]:
# ============================================================
# Load SS pickles + merge if tracking_only + mapping_change_only
# ============================================================

NEED_MERGE = (B_MP1_TYPE == 'tracking_only' and
              B_MP2_TYPE in ('mapping_change_only', 'mapping_change'))

print(f"Loading {mp1_label}...")
n_data_mp1_raw, f_data_mp1_raw = load_pickled_ss("SKIEUR_hs_0", B_MP1_TYPE, dt)
print(f"  Loaded {len(n_data_mp1_raw)} sessions")

print(f"Loading {mp2_label}...")
n_data_mp2_raw, f_data_mp2_raw = load_pickled_ss("SKIEUR_hs_0", B_MP2_TYPE, dt)
print(f"  Loaded {len(n_data_mp2_raw)} sessions")

if NEED_MERGE:
    # ── Pair by Google Sheet "paired_sessions" column ──
    # Sheet row: session=SESSION_00, paired_sessions=SESSION_01 (the matching MP2)
    sheet_mp1 = df_sheet[df_sheet["type"] == B_MP1_TYPE].copy()
    
    pairs = []
    for _, row in sheet_mp1.iterrows():
        mp1_name = row["session_name"]
        mp2_name = str(row.get("paired_sessions", "")).strip()
        if mp2_name and mp2_name != 'nan':
            # Verify mp2_name is actually in the MP2 type
            if mp2_name in df_sheet[df_sheet["type"] == B_MP2_TYPE]["session_name"].values:
                pairs.append((mp1_name, mp2_name))
            else:
                print(f"  WARNING: paired_sessions={mp2_name} not found in {B_MP2_TYPE} — skipped")
        else:
            print(f"  WARNING: {mp1_name} has no paired_sessions — skipped (orphan)")

    print(f"\\n  Pairing from Sheet: {len(pairs)} valid pairs (via paired_sessions column)")
    for n1, n2 in pairs:
        print(f"    {n1} + {n2}")

    # Filter pickle data to only sessions in valid pairs
    names_mp1 = df_sheet[df_sheet["type"] == B_MP1_TYPE]["session_name"].tolist()
    names_mp2 = df_sheet[df_sheet["type"] == B_MP2_TYPE]["session_name"].tolist()
    mp1_keep_idx = [names_mp1.index(n1) for n1, _ in pairs
                    if n1 in names_mp1 and names_mp1.index(n1) < len(n_data_mp1_raw)]
    mp2_keep_idx = [names_mp2.index(n2) for _, n2 in pairs
                    if n2 in names_mp2 and names_mp2.index(n2) < len(n_data_mp2_raw)]
    n_min = min(len(mp1_keep_idx), len(mp2_keep_idx))
    mp1_keep_idx = mp1_keep_idx[:n_min]
    mp2_keep_idx = mp2_keep_idx[:n_min]

    n_data_mp1 = [n_data_mp1_raw[i] for i in mp1_keep_idx]
    f_data_mp1 = [f_data_mp1_raw[i] for i in mp1_keep_idx]
    n_data_mp2 = [n_data_mp2_raw[i] for i in mp2_keep_idx]
    f_data_mp2 = [f_data_mp2_raw[i] for i in mp2_keep_idx]

    # Merge MP1 + MP2 along time axis
    print(f"\\n  MERGING {len(n_data_mp1)} paired sessions...")
    n_data_merged, f_data_merged = [], []
    for i in range(len(n_data_mp1)):
        assert n_data_mp1[i].shape[0] == n_data_mp2[i].shape[0], \
            f"Neuron count mismatch at pair {i}"
        n_merged = np.concatenate([n_data_mp1[i], n_data_mp2[i]], axis=1)
        f_data_mp1[i]['mapping_type'] = 'MP1'
        f_data_mp2[i]['mapping_type'] = 'MP2'
        for fd, st in [(f_data_mp1[i], B_MP1_TYPE), (f_data_mp2[i], B_MP2_TYPE)]:
            fd['session_type'] = st
            if 'Condition' in fd.columns:
                fd['condition_label'] = fd['Condition'].map(
                    {0.0: 'Tracking', 1.0: 'Playback', 0: 'Tracking', 1: 'Playback'}).fillna('Unknown')
        f_data_merged.append(pd.concat([f_data_mp1[i], f_data_mp2[i]], ignore_index=True))
        n_data_merged.append(n_merged)

    n_data_all = n_data_merged
    f_data_all = f_data_merged
    MERGED = True
    print_classification(f_data_all, f"Merged ({len(pairs)} pairs)")

else:
    f_data_mp1 = label_sessions(f_data_mp1_raw, 'MP1', B_MP1_TYPE)
    f_data_mp2 = label_sessions(f_data_mp2_raw, 'MP2', B_MP2_TYPE)
    FORCE_EQUAL = 1
    if FORCE_EQUAL:
        n_min = min(len(f_data_mp1), len(f_data_mp2))
        n_data_mp1 = n_data_mp1_raw[:n_min]; f_data_mp1 = f_data_mp1[:n_min]
        n_data_mp2 = n_data_mp2_raw[:n_min]; f_data_mp2 = f_data_mp2[:n_min]
        print(f"FORCE_EQUAL=1: trimmed to {n_min} sessions each")
    n_data_all = None; f_data_all = None; MERGED = False
    print_classification(f_data_mp1 + f_data_mp2, "Separate")
    print(f"\\n  NOT MERGED: independent session sets.")


In [ ]:
# ============================================================
# Session list: Beginner/Expert split
# ============================================================

if MERGED:
    # Merged: one list of paired sessions
    n_sess = len(n_data_all)
    n_half = n_sess // 2
    sessions_mp1 = df_sheet[df_sheet["type"] == B_MP1_TYPE]["session_name"].tolist()[:n_sess]
    sessions_mp2 = df_sheet[df_sheet["type"] == B_MP2_TYPE]["session_name"].tolist()[:n_sess]
    print(f"  Merged pairs ({n_sess} sessions):")
    print(f"    Beginner ({n_half}):")
    for i in range(n_half):
        print(f"      [{i}] {sessions_mp1[i]} + {sessions_mp2[i]}")
    print(f"    Expert ({n_sess - n_half}):")
    for i in range(n_half, n_sess):
        print(f"      [{i}] {sessions_mp1[i]} + {sessions_mp2[i]}")
else:
    for mp_label, n_data_list, stype in [
        (mp1_label, n_data_mp1, B_MP1_TYPE),
        (mp2_label, n_data_mp2, B_MP2_TYPE),
    ]:
        n_sess = len(n_data_list)
        n_half = n_sess // 2
        sessions = df_sheet[df_sheet["type"] == stype]["session_name"].tolist()[:n_sess]
        print(f"  {mp_label} ({stype}): {n_sess} sessions")
        print(f"    Beginner ({n_half}): {sessions[:n_half]}")
        print(f"    Expert   ({n_sess - n_half}): {sessions[n_half:]}")
        print()


---
## Build Trajectories

H1/H2 PSTH trajectories, baseline-subtracted, averaged across triggers.


In [ ]:
# ============================================================
# Build trajectories
# ============================================================

if MERGED:
    # Merged mode: data already concatenated along time, mapping_type in features.
    # Build trajectories from merged data; mapping_type distinguishes MP1 from MP2.
    n_data_proc = remove_average(smooth_data(n_data_all))
    n_data_list, f_data_list = re_organise_data([n_data_proc], [f_data_all])
    n_before = len(n_data_list)
    n_data_list, f_data_list = zip(*[(n, f) for n, f in zip(n_data_list, f_data_list)
                                      if abs(n.shape[-1] - len(f)) <= 2])
    n_data_list, f_data_list = list(n_data_list), list(f_data_list)
    if len(n_data_list) != n_before:
        print(f"  Shape-filter dropped {n_before - len(n_data_list)} sessions")

    n_half = len(n_data_list) // 2
    beg_n, exp_n = list(n_data_list[:n_half]), list(n_data_list[n_half:])
    beg_f, exp_f = list(f_data_list[:n_half]), list(f_data_list[n_half:])

    # Build separate trajectories per mapping_type so downstream LMM can use them
    traj_data = {}
    for mt in ['MP1', 'MP2']:
        traj = {}
        for q in [(0.0, 0.5), (0.5, 1.0)]:
            track_list, pb_list = [], []
            for nd, fd in zip(beg_n + exp_n, beg_f + exp_f):
                mask = fd['mapping_type'] == mt
                if not mask.any():
                    continue
                fd_sub = fd[mask].reset_index(drop=True)
                r = extract_traj_subset([nd], [fd_sub], t_pre, t_post, dt,
                                         overlap_thresh=1.0, n_pre=n_pre, full=True,
                                         trial_start=q[0], trial_end=q[1])
                track_list.extend(r[0])
                pb_list.extend(r[1])
            traj[q] = {'track': track_list, 'pb': pb_list}
        traj_data[mt] = traj
        n_tr = sum(1 for t in traj[(0.0, 0.5)]['track'] if t is not None)
        n_pb = sum(1 for t in traj[(0.0, 0.5)]['pb'] if t is not None)
        print(f"  Merged {mt}: {n_tr} TR + {n_pb} PB traj")

    traj_by_half_mp1 = traj_data['MP1']
    traj_by_half_mp2 = traj_data['MP2']
    print(f"\\ntraj_by_half_mp1 (MP1), traj_by_half_mp2 (MP2) ready (MERGED).")
    print("  NOTE: mapping_type is within-session for merged pairs.")

else:
    # Separate mode: MP1 and MP2 are independent session sets
    n_data_mp1_proc = remove_average(smooth_data(n_data_mp1))
    n_data_mp2_proc = remove_average(smooth_data(n_data_mp2))

    data_pairs = [
        (mp1_label, (n_data_mp1_proc, f_data_mp1)),
        (mp2_label, (n_data_mp2_proc, f_data_mp2)),
    ]
    traj_data = {}
    for mp_label, (n_data_list, f_data_list) in data_pairs:
        n_data_list, f_data_list = re_organise_data([n_data_list], [f_data_list])
        n_before = len(n_data_list)
        n_data_list, f_data_list = zip(*[(n, f) for n, f in zip(n_data_list, f_data_list)
                                          if abs(n.shape[-1] - len(f)) <= 2])
        n_data_list, f_data_list = list(n_data_list), list(f_data_list)
        n_dropped = n_before - len(n_data_list)
        if n_dropped > 0:
            print(f"  {mp_label}: shape-filter dropped {n_dropped}/{n_before} sessions")

        n_half = len(n_data_list) // 2
        beg_n, exp_n = list(n_data_list[:n_half]), list(n_data_list[n_half:])
        beg_f, exp_f = list(f_data_list[:n_half]), list(f_data_list[n_half:])

        traj = {}
        for q in [(0.0, 0.5), (0.5, 1.0)]:
            r_beg = extract_traj_subset(beg_n, beg_f, t_pre, t_post, dt,
                                         overlap_thresh=1.0, n_pre=n_pre, full=True,
                                         trial_start=q[0], trial_end=q[1])
            r_exp = extract_traj_subset(exp_n, exp_f, t_pre, t_post, dt,
                                         overlap_thresh=1.0, n_pre=n_pre, full=True,
                                         trial_start=q[0], trial_end=q[1])
            tb, pb, *_ = r_beg
            te, pe, *_ = r_exp
            traj[q] = {'track': {'beg': tb, 'exp': te},
                       'pb':    {'beg': pb, 'exp': pe}}
        traj_data[mp_label] = traj
        print(f"  {mp_label}: {n_half*2} sessions -> {len(tb)} TR + {len(pb)} PB traj")

    traj_by_half_mp1 = traj_data[mp1_label]
    traj_by_half_mp2 = traj_data[mp2_label]
    print(f"\\ntraj_by_half_mp1 ({mp1_label}), traj_by_half_mp2 ({mp2_label}) ready (SEPARATE).")
    print("  NOTE: mapping_type is between-session for separate sets.")


---
## Quick Tests + LMM

Identical to `Skieur_Full_LMM.ipynb` sections 2.1–2.5.


In [ ]:
# ===============================================================
# Quick tests: Track vs PB, MP1 vs MP2
# ===============================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import build_lmm_dataframe
from scipy.stats import ttest_rel, ttest_ind

all_p_vals_anova = []
all_p_labels_anova = []

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        print(f"\n{'='*60}")
        print(f"  METRIC: {metric}  |  WINDOW: {label}")
        print("="*60)

        df1 = build_lmm_dataframe(traj_by_half_mp1, "MP1", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df2 = build_lmm_dataframe(traj_by_half_mp2, "MP2", time,
                                   auc_t_start=0.0, auc_t_end=t_window,
                                   split_half=False, metric=metric)
        df = pd.concat([df1, df2], ignore_index=True)

        tr = df[df["condition"] == "Track"]["response"].values
        pb = df[df["condition"] == "Playback"]["response"].values
        t_tr_pb, p_tr_pb = ttest_rel(pb, tr)
        print(f"  Track vs Playback (paired): t={t_tr_pb:.3f}, p={p_tr_pb:.4e}")
        all_p_vals_anova.append(p_tr_pb)
        all_p_labels_anova.append(f"Track-vs-PB_{metric}_{label}")

        df_neuron = df.groupby(["neuron_id", "mapping"])["response"].mean().reset_index()
        mp1_vals = df_neuron[df_neuron["mapping"] == "MP1"]["response"].values
        mp2_vals = df_neuron[df_neuron["mapping"] == "MP2"]["response"].values
        t_mp, p_mp = ttest_ind(mp1_vals, mp2_vals, equal_var=False)
        print(f"  MP1 vs MP2 (Welch):        t={t_mp:.3f}, p={p_mp:.4e}")
        all_p_vals_anova.append(p_mp)
        all_p_labels_anova.append(f"MP1-vs-MP2_{metric}_{label}")

# Global FDR
from statsmodels.stats.multitest import multipletests
if all_p_vals_anova:
    reject, p_fdr, _, _ = multipletests(all_p_vals_anova, method='fdr_bh')
    print(f"\n{'='*70}")
    print("  QUICK TESTS: GLOBAL FDR (BH)")
    print("="*70)
    for label, p_raw, p_corr in zip(all_p_labels_anova, all_p_vals_anova, p_fdr):
        sig = '***' if p_corr<0.001 else ('**' if p_corr<0.01 else ('*' if p_corr<0.05 else 'n.s.'))
        print(f"  {label:<40s} raw={p_raw:.6f}  fdr={p_corr:.6f}  {sig}")


In [ ]:
# ================================================================
# Full LMM: condition * mapping * expertise * half
# ================================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

for metric in ["mean", "peak"]:
    for t_window, label in [(0.1, "0-100ms"), (0.3, "0-300ms")]:
        print(f"\n{'='*70}")
        print(f"  METRIC: {metric}  |  WINDOW: {label}")
        print("="*70)
        result_lmm, df_lmm = run_lmm_analysis(
            traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
            auc_t_start=0.0, auc_t_end=t_window, metric=metric
        )
        compare_random_effect_structures(df_lmm)


In [ ]:
# ================================================================
# Peak 0-200ms: Full LMM
# ================================================================
import importlib
import lmm_analysis
importlib.reload(lmm_analysis)
from lmm_analysis import run_lmm_analysis, compare_random_effect_structures

result_peak, df_peak = run_lmm_analysis(
    traj_by_half_mp1, traj_by_half_mp2, time, save_directory,
    auc_t_start=0.0, auc_t_end=0.2, metric="peak"
)
compare_random_effect_structures(df_peak)
